# Off-chain pilot: public token metadata and evidence-level linkage

**Lead:** Shilin. **Independent coauthor reproduction:** Claire, pending. **Pilot date:** 17 September 2026 UTC. This editable Colab uses real Pump.fun creation URIs from Claire's pinned on-chain data and a bounded live metadata request. It also replays a fixed, rights-limited output snapshot so results remain inspectable if a gateway changes. Start with a fresh CPU runtime and run in order.

[Proposal and editable Mermaid diagrams](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/PROPOSAL.md) · [Data dictionary](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/DATA_DICTIONARY.md) · [Code](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1) · [Fixed release](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/release) · [Claire source](https://huggingface.co/datasets/global-nomad-nexus/claire-threechain-v1/tree/8b29598a6565b67a8a943962dbf77f3d6b2559de). The complete source receipts, method, and actual validation are in the linked release. The fixed release excludes third-party response bodies because redistribution rights have not been cleared.

<a id="part-1"></a>
## Part 1 — Overview and navigation

A blockchain creation record may contain a metadata URI. A URI is a pointer, not the metadata itself. We request the exact URI, preserve the response time and hash, parse named JSON fields, then make field-level assertions tied to both the chain event and retrieved response. A website or social link is a declaration in that response, not proof of account ownership or presence at creation time.

1. [Overview](#part-1)  2. [Sources and dictionary](#part-2)  3. [Acquisition](#part-3)  4. [Inspect acquired records](#part-4)  5. [Processing](#part-5)  6. [Inspect processed records](#part-6)  7. [Descriptive reuse](#part-7)  8. [Validation and handoff](#part-8).

Run each code cell after reading its preceding explanation. The fixed snapshot is approximately 250 KB, requires no login or GPU, and is suitable for a fresh Colab. Live gateway access can vary; the notebook reports that result separately.

<a id="part-2"></a>
## Part 2 — Sources, metadata and data dictionary

**Sampling frame.** Claire's immutable three-chain snapshot covers 2026-09-14 12:00:00 ≤ block time < 12:05:00 UTC. We select committed, decoded, nonempty-object creation records, yielding 116 events: 61 Pump.fun on Solana, 55 Four.meme on BSC, and zero recognized Clanker creations on Base. Events and distinct chain-qualified tokens are both counted; this is a five-minute engineering pilot, not a representative launch sample.

**Access.** Pump.fun creation events supply exact metadata URIs. Original URIs often use `https://ipfs.io/ipfs/<CID>`; the collection tried that gateway and then `https://gateway.pinata.cloud/ipfs/<CID>` on HTTP 429. The live cell requests one exact CID through Pinata with a declared 30-second timeout. [Collection code](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/collect_metadata.py) preserves attempts, response status, retrieval UTC, byte count and SHA-256. Four.meme's documented exact-address endpoint `https://four.meme/meme-api/v1/private/token/get/v2?address=<contract>` returned HTTP 403 in three spaced probes; those requests and 52 unprobed BSC events remain visible. No API key is used.

**Units and key fields.** `onchain_launch_cohort`: creation event, `launch_record_id` primary key, `object_id` chain-qualified token, `chain_event_time_utc`, `metadata_uri_declared`. `offchain_snapshots`: one successful response per URI, `snapshot_id`, `retrieved_at_utc`, `raw_sha256`, `parse_status`. `offchain_declarations`: one JSON field URL, `snapshot_id`, `field_name`, `json_pointer`, `target_class`. `coverage_ledger`: one row per event including failures. Full types, null semantics, keys and rights are in the [dictionary](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/DATA_DICTIONARY.md) and [rights register](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/rights_sources.csv). Source revision and hashes are in `release_manifest.json`.

**Diagram.** <img src="https://raw.githubusercontent.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/figures/dgp.svg" width="900" alt="Editable data generating process diagram" />

```mermaid
flowchart LR
 A[Creator enters token details] --> B[Launch platform]
 B --> C[On-chain creation event]
 B -. may declare .-> D[Metadata URI]
 D --> E[Retrieved JSON response]
 E --> F[Website or social URL field]
 C --> G[Evidence-backed assertion]
 F --> G
```

The solid arrows are recorded activities or evidence paths; the dotted arrow is contingent. Chain time and retrieval time are different observations. This Mermaid source and the full editable diagram in the proposal can be changed by both authors.

In [ ]:
import os, sys, json, hashlib, tarfile, tempfile, urllib.request
from collections import Counter
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

CODE_COMMIT = "1fec501de1de09d9cc2b9c69ce350338a40af889"
EXPECTED_ARROW = "25.0.1"
if pa.__version__ != EXPECTED_ARROW:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"pyarrow=={EXPECTED_ARROW}"])
    print("Restart the runtime, then run from the first cell to use the pinned PyArrow version.")
    raise SystemExit(0)

local = os.environ.get("PILOT_LOCAL_REPO")
if local:
    ROOT = Path(local).expanduser().resolve()
    print("Local test checkout:", ROOT)
else:
    url = f"https://codeload.github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/tar.gz/{CODE_COMMIT}"
    request = urllib.request.Request(url, headers={"User-Agent": "shilin-offchain-pilot-colab/1.0"})
    body = urllib.request.urlopen(request, timeout=90).read()
    tmp = Path(tempfile.mkdtemp(prefix="shilin-pilot-"))
    archive = tmp / "repo.tar.gz"
    archive.write_bytes(body)
    with tarfile.open(archive, "r:gz") as tar:
        names = tar.getnames()
        prefix = names[0].split("/")[0]
        tar.extractall(tmp, filter="data")
    ROOT = tmp / prefix
    print("Downloaded pinned code and release:", CODE_COMMIT)

PILOT = ROOT / "pilots" / "shilin-offchain-v1"
RELEASE = PILOT / "release"
sys.path.insert(0, str(PILOT))
from verify_release import verify
verification = verify(RELEASE)
assert verification["passed"], verification
manifest = json.loads((RELEASE / "release_manifest.json").read_text())
print("Fixed release verified:", verification)
print("PyArrow:", pa.__version__)
print("Claire input revision:", manifest["claire_hf_revision"])

<a id="part-3"></a>
## Part 3 — Query or acquire real data

The original acquisition processed **57 distinct URIs in 94 HTTP attempts**. It saved local response bytes and a request ledger; this public release keeps hashes, request metadata and extracted factual fields. The cell below makes a fresh real HTTP request for one CIDv1 item from the cohort. It prints current status, size, SHA-256 and three covered fields. A changed response or failed gateway is evidence about **today's** access, not a reason to rewrite the fixed 17 September snapshot. [Acquisition code](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/collect_metadata.py).

In [ ]:
from urllib.error import HTTPError, URLError
from datetime import datetime, timezone
LIVE_URL = "https://gateway.pinata.cloud/ipfs/bafkreih4am23irlugdsonanclckaxerem23hznxn5tl424ji3rokqzq4ze"
live_result = {"url": LIVE_URL, "checked_at_utc": datetime.now(timezone.utc).isoformat()}
try:
    request = urllib.request.Request(LIVE_URL, headers={"User-Agent": "shilin-offchain-pilot-colab/1.0", "Accept": "application/json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        raw = response.read(2_000_001)
        live_result.update(status=response.status, bytes=len(raw), sha256=hashlib.sha256(raw).hexdigest())
        if len(raw) > 2_000_000: raise ValueError("Live response exceeds 2 MB guard")
        document = json.loads(raw)
        live_result["covered_fields"] = {key: document.get(key) for key in ("website", "twitter", "telegram")}
except (HTTPError, URLError, TimeoutError, ValueError, json.JSONDecodeError) as exc:
    live_result["error"] = str(exc)
print(json.dumps(live_result, indent=2))
assert live_result.get("status") == 200 or "error" in live_result

<a id="part-4"></a>
## Part 4 — Inspect acquired records

Read the fixed request receipt and cohort before any link analysis. The original 94 attempts include failed first-gateway responses; 57 distinct URI response snapshots succeeded. HTTP failures are not missing records in the denominator and do not imply no online presence. Compare the live one-row result above to the fixed snapshot only as a current re-fetch check.

In [ ]:
from collections import Counter
requests = [json.loads(line) for line in (RELEASE / "offchain_requests.jsonl").read_text().splitlines() if line]
cohort = pq.read_table(RELEASE / "onchain_launch_cohort.parquet").to_pylist()
snapshots = pq.read_table(RELEASE / "offchain_snapshots.parquet").to_pylist()
print("Creation events by platform:", dict(Counter(x["platform_id"] for x in cohort)))
print("HTTP attempts/status:", len(requests), dict(Counter(str(x.get("status_code")) for x in requests)))
print("Successful parsed response snapshots:", len(snapshots), dict(Counter(x["parse_status"] for x in snapshots)))
print("First cohort record:", {k:cohort[0].get(k) for k in ("launch_record_id","object_id","chain_event_time_utc","metadata_uri_declared")})
print("First response:", {k:snapshots[0][k] for k in ("snapshot_id","retrieved_at_utc","raw_sha256","parse_status")})
frozen_example = next(x for x in snapshots if x["snapshot_id"] == "snap:f5b91dff52b0588cd2795b4e")
print("Same bytes as 17 Sep snapshot?", live_result.get("sha256") == frozen_example["raw_sha256"] if "sha256" in live_result else "live fetch unavailable")

<a id="part-5"></a>
## Part 5 — Process data and document decisions

[Processing code](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/process_linkage.py) parses JSON objects, reads only nonempty `website`, `twitter`, and `telegram` URL fields, normalizes URLs, classifies targets, and attaches each value to an exact response snapshot and on-chain URI assertion. A social post in a `website` field retains the field label and receives `target_class=social_post`; it is never promoted to a verified website. Duplicate creation records are removed by deterministic event key in [cohort code](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/build_cohort.py); shared URIs produce one response snapshot but retain their event-level joins. No fuzzy name or ticker match is accepted.

<img src="https://raw.githubusercontent.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/figures/pipeline.svg" width="1000" alt="Editable acquisition and processing pipeline" />

```mermaid
flowchart LR
 A[Claire pinned decoded records] --> B[Creation cohort]
 B --> C[Exact URI requests]
 C --> D[Receipt and response hash]
 D --> E[JSON field extraction]
 E --> F[Candidate + evidence + assertion]
 F --> G[Coverage and validation]
```

Each stage's input, output, filter and check is in the [proposal stage map](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/PROPOSAL.md). Raw bytes remain local pending source rights review; public fixed tables permit integration replay, while live re-fetch permits a current parsing demonstration.

In [ ]:
declarations = pq.read_table(RELEASE / "offchain_declarations.parquet").to_pylist()
evidence = pq.read_table(RELEASE / "linkage_evidence.parquet").to_pylist()
assertions = pq.read_table(RELEASE / "linkage_assertions.parquet").to_pylist()
coverage = pq.read_table(RELEASE / "coverage_ledger.parquet").to_pylist()
print("Declaration fields:", dict(Counter(x["field_name"] for x in declarations)))
print("URL target classes:", dict(Counter(x["target_class"] for x in declarations)))
print("Coverage states:", dict(Counter(x["coverage_state"] for x in coverage)))
print("Evidence and assertions:", len(evidence), len(assertions))
assert len(coverage)==len(cohort)==116
assert sum(x["coverage_state"]=="declaration_observed" for x in coverage)==29

<a id="part-6"></a>
## Part 6 — Inspect processed evidence

Follow one actual assertion across the event, response and JSON pointer. The on-chain event fixes the token and declared URI at creation. The later response provides the field value. It does **not** prove the destination page was live at creation. The manually reviewed case and a misleading website-field example are recorded in `REVIEW_EXAMPLES.md`.

In [ ]:
case = next(x for x in assertions if x["assertion_id"] == "assert:136781253019bac2c496113b")
case_evidence = [x for x in evidence if x["evidence_id"] in case["evidence_ids"]]
case_snapshot = next(x for x in snapshots if x["snapshot_id"] == "snap:f5b91dff52b0588cd2795b4e")
print("Assertion:", {k:case[k] for k in ("object_id","relation_type","right_value","chain_event_time_utc","first_verified_at_utc","as_of_eligibility")})
print("Evidence:", [{k:x[k] for k in ("evidence_kind","raw_ref","snapshot_id","field_pointer")} for x in case_evidence])
print("Snapshot:", {k:case_snapshot[k] for k in ("original_uri","retrieved_at_utc","raw_sha256","cid_integrity_status")})
misleading = next(x for x in assertions if x["assertion_id"] == "assert:b7f1d653c3ab85ec9143ae43")
print("Field/target mismatch:", misleading["relation_type"], misleading["right_value"])
assert case["as_of_eligibility"] == misleading["as_of_eligibility"] == "unknown"

<a id="part-7"></a>
## Part 7 — Small descriptive example and reuse

The unit below is an eligible creation event. Report how many have at least one covered link field in a retrieved JSON object, while keeping all 116 events in the denominator. This is **source coverage**, not a measure of project quality, adoption, identity or token success. The five-minute window and source access conditions limit generalization.

The [DIVE Data Descriptor](https://doi.org/10.1038/s41597-026-07025-5) informs documentation and measured technical validation; [Multi-Chain Graphs of Graphs](https://proceedings.neurips.cc/paper_files/paper/2024/file/3205b048f9cc54b9f7963db0b0f52d53-Paper-Datasets_and_Benchmarks_Track.pdf) §3.2 motivates explicit chain-specific sampling and source provenance. Their methods and time spans are not claimed as our pilot results.

In [ ]:
by_platform = {}
for platform in sorted({x["platform_id"] for x in coverage}):
    rows = [x for x in coverage if x["platform_id"] == platform]
    by_platform[platform] = {"events": len(rows), "declared_field_events": sum(x["coverage_state"] == "declaration_observed" for x in rows), "states": dict(Counter(x["coverage_state"] for x in rows))}
print(json.dumps(by_platform, indent=2))
print("Overall observed field declarations:", sum(x["coverage_state"] == "declaration_observed" for x in coverage), "/", len(coverage))
assert by_platform["pump.fun"]["declared_field_events"] == 29

<a id="part-8"></a>
## Part 8 — Technical validation and coauthor handoff

Run the public fixed-release verifier below. The original source-side validation checks response hashes against saved local bytes, three sampled raw Pump event re-decodes, foreign keys, denominator, CIDv1 raw SHA-256, and time rules; its measured report is [here](https://github.com/Global-Nomad-Nexus/Web3AI4IO-SD-Template/blob/1fec501de1de09d9cc2b9c69ce350338a40af889/pilots/shilin-offchain-v1/release/validation_report.md). This public verification checks published file hashes, row counts, keys and temporal flags. The fixed release has 57 JSON snapshots, 39 URL field declarations and 100 evidence-backed assertions. CIDv0 DAG-PB content was not cryptographically verified, and BSC API access was restricted here.

**Coauthor review requested:** Claire should run this notebook in a fresh runtime, inspect at least one raw event-to-URI path and one field-to-response path, record runtime/version and any disagreement, then send corrections before a coordinated submission. This execution alone is not that independent review. A formal Data Descriptor also needs a larger justified cohort, durable archive/rights decision and further validation.

In [ ]:
result = verify(RELEASE)
print(json.dumps(result, indent=2))
source_validation = json.loads((RELEASE / "validation.json").read_text())
print("Source-side validation:", source_validation.get("overall_status", source_validation.get("passed")))
assert result["passed"] and result["cohort_events"] == 116
print("Completed fixed-snapshot off-chain tutorial. Record live_result separately in any review receipt.")